# Notebook 5: Retention Strategy & Business Recommendations

---

## Business Question

**What specific actions should we take to reduce churn and maximize customer value?**

I've done the analysis. I've built the models. Now comes the most important part: translating insights into action. What should the business actually do with what I've learned?

## Why This Matters

Analysis without action is just intellectual exercise. The goal is to provide recommendations that:
- Are specific enough to implement
- Have clear expected impact
- Consider trade-offs and limitations

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../data/model_predictions.csv')
print(f"Loaded {len(df):,} customers with model predictions")

---

## Executive Summary

Before diving into specific recommendations, let me recap where we are.

---

In [ ]:
# Key metrics
total_customers = len(df)
churned = df['churned'].sum()
churn_rate = df['churned'].mean() * 100
total_revenue = df['monthly_fee'].sum()
revenue_lost = df[df['churned'] == 1]['monthly_fee'].sum()

# High-risk customers (not churned)
high_risk = df[(df['risk_category'].isin(['High', 'Critical'])) & (df['churned'] == 0)]

print("="*60)
print("CURRENT STATE SUMMARY")
print("="*60)
print(f"\nCustomer Base: {total_customers:,}")
print(f"Churn Rate: {churn_rate:.1f}%")
print(f"Monthly Revenue: ${total_revenue:,.0f}")
print(f"Revenue Lost to Churn: ${revenue_lost:,.0f}/month")
print(f"\nHigh-Risk Customers (Active): {len(high_risk):,}")
print(f"Revenue at Risk: ${high_risk['monthly_fee'].sum():,.0f}/month")

---

## Segment-Specific Retention Strategies

Different customers need different approaches. Here's my recommended strategy by segment.

---

In [ ]:
# Segment summary
segment_summary = df.groupby('segment_name').agg({
    'customer_id': 'count',
    'churned': 'mean',
    'monthly_fee': ['mean', 'sum'],
    'health_score': 'mean',
    'clv_calculated': 'mean'
}).round(2)

segment_summary.columns = ['customers', 'churn_rate', 'avg_fee', 'total_revenue', 'avg_health', 'avg_clv']
segment_summary['churn_rate'] = (segment_summary['churn_rate'] * 100).round(1)

print("Segment Summary:")
print(segment_summary)

### Strategy 1: At-Risk Segment - Immediate Intervention

**Problem:** These customers show warning signs. Many haven't logged in for 30+ days.

**Goal:** Win them back before they cancel.

---

In [ ]:
at_risk = df[df['segment_name'] == 'At-Risk']

print(f"At-Risk Segment Analysis:")
print(f"  Customers: {len(at_risk):,}")
print(f"  Current Churn Rate: {at_risk['churned'].mean()*100:.1f}%")
print(f"  Average Health Score: {at_risk['health_score'].mean():.0f}")
print(f"  Revenue at Stake: ${at_risk['monthly_fee'].sum():,.0f}/month")

In [ ]:
# Recommended actions
print("\n" + "="*60)
print("RECOMMENDED ACTIONS FOR AT-RISK SEGMENT")
print("="*60)

print("""
1. WIN-BACK EMAIL CAMPAIGN
   - Trigger: Customer hasn't logged in for 14+ days
   - Offer: 20% discount for 3 months OR 1 free month
   - Personalization: Highlight shows similar to their watch history
   - Expected impact: 10-15% conversion

2. PERSONAL OUTREACH (High-value customers only)
   - Target: Customers with monthly_fee >= $15 AND health_score < 50
   - Action: Personal email from customer success team
   - Ask: "What would make you stay?"
   - Expected impact: Higher engagement from VIP customers

3. SURVEY AT CANCELLATION
   - Capture exit reason for every churned customer
   - Use data to improve product and future predictions
   - Offer last-minute retention deal based on reason

4. CONTENT RECOMMENDATIONS
   - Push notification with "New shows you might like"
   - Based on viewing history and genre preferences
   - Timing: Evenings when people typically watch
""")

# Estimate impact
active_at_risk = at_risk[at_risk['churned'] == 0]
if len(active_at_risk) > 0:
    expected_save = int(len(active_at_risk) * 0.15)
    revenue_save = expected_save * active_at_risk['monthly_fee'].mean()
    print(f"\nESTIMATED IMPACT (15% save rate):")
    print(f"  Customers saved: ~{expected_save:,}")
    print(f"  Monthly revenue recovered: ~${revenue_save:,.0f}")

### Strategy 2: Casual Segment - Engagement Focus

**Problem:** These customers are moderately engaged but could drift away.

**Goal:** Increase engagement before they become at-risk.

---

In [ ]:
casual = df[df['segment_name'] == 'Casual']

print(f"Casual Segment Analysis:")
print(f"  Customers: {len(casual):,}")
print(f"  Churn Rate: {casual['churned'].mean()*100:.1f}%")
print(f"  Average Watch Hours: {casual['watch_hours'].mean():.1f}")
print(f"  Average Health Score: {casual['health_score'].mean():.0f}")

In [ ]:
print("\n" + "="*60)
print("RECOMMENDED ACTIONS FOR CASUAL SEGMENT")
print("="*60)

print("""
1. PERSONALIZED CONTENT RECOMMENDATIONS
   - Weekly email with "Recommended for You" section
   - Based on genre preferences and watch history
   - Include trending shows in their preferred categories

2. ENGAGEMENT MILESTONES
   - Gamification: "You've watched X hours this month!"
   - Badges for completing series or trying new genres
   - Social features: Share what you're watching

3. PROFILE EXPANSION PROMPTS
   - Many casual users have only 1-2 profiles
   - Prompt: "Add family members for a better experience"
   - Family plans increase retention (shared investment)

4. LIMITED-TIME UPGRADE OFFERS
   - For engaged casual users: Upgrade to Standard at 20% off
   - Position as: "Unlock HD streaming and more profiles"
   - Upsell increases commitment and value
""")

### Strategy 3: Premium Segment - Value Maximization

**Problem:** High-paying customers who might not be fully utilizing their subscription.

**Goal:** Ensure they feel they're getting value for money.

---

In [ ]:
premium = df[df['segment_name'] == 'Premium']

print(f"Premium Segment Analysis:")
print(f"  Customers: {len(premium):,}")
print(f"  Churn Rate: {premium['churned'].mean()*100:.1f}%")
print(f"  Average Fee: ${premium['monthly_fee'].mean():.2f}")
print(f"  Total Revenue: ${premium['monthly_fee'].sum():,.0f}/month")

In [ ]:
print("\n" + "="*60)
print("RECOMMENDED ACTIONS FOR PREMIUM SEGMENT")
print("="*60)

print("""
1. EXCLUSIVE EARLY ACCESS
   - Premium members get early access to new releases
   - Creates feeling of VIP status
   - Differentiates from lower tiers

2. FAMILY PLAN UPSELL
   - Many Premium users have unused profile capacity
   - Promote adding family members
   - "Your family can enjoy Premium too!"

3. BEHIND-THE-SCENES CONTENT
   - Exclusive documentaries, director's cuts
   - Makes Premium feel special
   - Content that can't be found elsewhere

4. PREMIUM SUPPORT
   - Dedicated support line for Premium customers
   - Faster response times
   - Shows commitment to their experience

5. ANNUAL SUBSCRIPTION DISCOUNT
   - Offer 2 months free for annual commitment
   - Reduces churn (customer is committed for a year)
   - Improves cash flow
""")

### Strategy 4: Power Users - Retention & Advocacy

**Problem:** Even highly engaged users can churn.

**Goal:** Turn them into brand advocates and keep them engaged.

---

In [ ]:
power = df[df['segment_name'] == 'Power Users']

print(f"Power Users Segment Analysis:")
print(f"  Customers: {len(power):,}")
print(f"  Churn Rate: {power['churned'].mean()*100:.1f}%")
print(f"  Average Watch Hours: {power['watch_hours'].mean():.1f}")
print(f"  Average Health Score: {power['health_score'].mean():.0f}")

In [ ]:
print("\n" + "="*60)
print("RECOMMENDED ACTIONS FOR POWER USERS")
print("="*60)

print("""
1. LOYALTY PROGRAM
   - Reward viewing milestones with perks
   - "Netflix Insider" status for top viewers
   - Exclusive merchandise or early access

2. REFERRAL PROGRAM
   - "Invite friends, get a free month"
   - Power users are likely to recommend anyway
   - Turn advocacy into tangible benefit

3. BETA ACCESS
   - Invite power users to test new features
   - They'll provide valuable feedback
   - Makes them feel valued and invested

4. VIP SUPPORT
   - Dedicated account manager for top customers
   - Immediate resolution of any issues
   - Prevents churn due to frustration

5. CONTENT INSIDER ACCESS
   - Behind-the-scenes content
   - Creator Q&As
   - Makes them feel connected to the platform
""")

---

## Revenue Impact Simulation

Let me model the potential revenue impact of different retention scenarios.

---

In [ ]:
def simulate_retention_impact(current_churn, target_churn, customers, avg_revenue):
    """Calculate revenue impact of reducing churn."""
    customers_saved = int(customers * (current_churn - target_churn))
    monthly_impact = customers_saved * avg_revenue
    annual_impact = monthly_impact * 12
    return customers_saved, monthly_impact, annual_impact

arpu = df['monthly_fee'].mean()

print("RETENTION IMPACT SIMULATION")
print("="*60)

for pct_reduction in [2, 5, 10]:
    target_churn = churn_rate/100 - pct_reduction/100
    if target_churn > 0:
        saved, monthly, annual = simulate_retention_impact(
            churn_rate/100, target_churn, total_customers, arpu
        )
        print(f"\n{pct_reduction}% churn reduction:")
        print(f"  Customers retained: {saved:,}")
        print(f"  Monthly revenue: ${monthly:,.0f}")
        print(f"  Annual revenue: ${annual:,.0f}")

In [ ]:
# Visualize retention scenarios
reductions = np.arange(1, 16) * 0.01
impacts = [simulate_retention_impact(churn_rate/100, churn_rate/100 - r, total_customers, arpu)
           for r in reductions]

annual_impacts = [i[2] for i in impacts]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([r*100 for r in reductions], annual_impacts, 'b-o', linewidth=2, markersize=6)
ax.fill_between([r*100 for r in reductions], annual_impacts, alpha=0.2)

ax.set_xlabel('Churn Reduction (%)', fontsize=12)
ax.set_ylabel('Annual Revenue Impact ($)', fontsize=12)
ax.set_title('Annual Revenue Impact by Churn Reduction Target', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Reference lines
ax.axvline(x=5, color='green', linestyle='--', alpha=0.7, label='5% reduction target')
ax.axvline(x=10, color='orange', linestyle='--', alpha=0.7, label='10% reduction target')
ax.legend()

plt.tight_layout()
plt.show()

---

## Priority Action Matrix

Not all actions are equal. Here's my prioritization based on impact vs effort.

---

In [ ]:
# Priority actions
actions = [
    {'Action': 'Automated re-engagement at 14-day inactivity', 
     'Impact': 'High', 'Effort': 'Low', 'Priority': 1,
     'Revenue_Impact': high_risk['monthly_fee'].sum() * 0.10},
    {'Action': 'Win-back email campaign for At-Risk segment', 
     'Impact': 'High', 'Effort': 'Low', 'Priority': 2,
     'Revenue_Impact': high_risk['monthly_fee'].sum() * 0.08},
    {'Action': 'Weekly health score updates to CS team', 
     'Impact': 'Medium', 'Effort': 'Low', 'Priority': 3,
     'Revenue_Impact': 1000},
    {'Action': 'Upsell Basic users with high engagement', 
     'Impact': 'High', 'Effort': 'Medium', 'Priority': 4,
     'Revenue_Impact': 3000},
    {'Action': 'Loyalty program for Power Users', 
     'Impact': 'Medium', 'Effort': 'Medium', 'Priority': 5,
     'Revenue_Impact': 1500},
    {'Action': 'Mobile app experience improvements', 
     'Impact': 'Medium', 'Effort': 'High', 'Priority': 6,
     'Revenue_Impact': 2000},
]

action_df = pd.DataFrame(actions)
print("PRIORITY ACTION MATRIX")
print("="*70)
print(action_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

action_df_sorted = action_df.sort_values('Priority')
colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#1abc9c', '#95a5a6']

bars = ax.barh(action_df_sorted['Action'], action_df_sorted['Revenue_Impact'], color=colors)

ax.set_xlabel('Estimated Monthly Revenue Impact ($)', fontsize=12)
ax.set_title('Retention Actions by Priority', fontsize=14, fontweight='bold')

for i, (bar, priority) in enumerate(zip(bars, action_df_sorted['Priority'])):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2, 
            f'Priority {priority}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---

## Implementation Roadmap

What should we do and when?

---

In [ ]:
print("="*70)
print("IMPLEMENTATION ROADMAP")
print("="*70)

print("""
WEEK 1-2: IMMEDIATE ACTIONS
────────────────────────────
□ Set up automated re-engagement triggers (14-day inactivity)
□ Deploy win-back email campaign for At-Risk segment
□ Begin weekly health score reports to CS team
□ Integrate churn risk scores into CRM

WEEK 3-4: SHORT-TERM ACTIONS
────────────────────────────
□ Launch personalized content recommendations
□ Implement exit survey at cancellation
□ Start upsell campaign for engaged Basic users
□ Set up customer health dashboard

MONTH 2-3: MEDIUM-TERM ACTIONS
────────────────────────────
□ Design and launch loyalty program
□ Implement referral program
□ A/B test retention offers
□ Build automated intervention workflows

MONTH 4+: LONG-TERM ACTIONS
────────────────────────────
□ Mobile app experience improvements
□ Regional content localization review
□ Advanced predictive model retraining
□ Customer success team training on health scores
""")

---

## Success Metrics

How will we know if our efforts are working?

---

In [ ]:
print("="*70)
print("KEY SUCCESS METRICS TO TRACK")
print("="*70)

print("""
PRIMARY METRICS
────────────────────────────
• Monthly churn rate
  Current: {:.1f}%  |  Target: {:.1f}%

• Revenue at risk
  Current: ${:,.0f}/month  |  Target: ${:,.0f}/month

• Average customer health score
  Current: {:.0f}  |  Target: {:.0f}

SECONDARY METRICS
────────────────────────────
• Win-back campaign conversion rate
• Re-engagement email open rate
• Upsell conversion rate
• Average engagement hours per customer
• Customer satisfaction (NPS) by segment
• Model AUC-ROC (monthly retraining)
""".format(
    churn_rate, churn_rate - 5,
    high_risk['monthly_fee'].sum(), high_risk['monthly_fee'].sum() * 0.7,
    df['health_score'].mean(), df['health_score'].mean() + 5
))

---

## Final Project Summary

---

In [ ]:
print("="*70)
print("PROJECT SUMMARY")
print("="*70)

print("""
PROBLEM
───────
~25% churn rate, significant monthly revenue loss

ROOT CAUSES IDENTIFIED
──────────────────────
1. Login recency is the strongest churn predictor (30+ days = high risk)
2. Basic tier has highest churn rate
3. Low engagement correlates with churn, but not perfectly
4. Mobile users churn at slightly higher rates

SOLUTIONS BUILT
───────────────
• Customer Segmentation (Power Users, Premium, Casual, At-Risk)
• Health Scoring System (0-100 scale)
• ML-based Churn Prediction (AUC-ROC ~0.74)
• Segment-specific retention strategies

EXPECTED BUSINESS IMPACT
───────────────────────
• 5% churn reduction = ~$38K annual revenue
• 10% churn reduction = ~$76K annual revenue
• High-risk customer identification enables targeted intervention

DELIVERABLES
────────────
✓ 5 Analysis Notebooks
✓ Trained ML Model (saved)
✓ Streamlit Dashboard
✓ SQL Business Queries
✓ Executive Summary Report

LIMITATIONS
───────────
• Cross-sectional data (no time series)
• No qualitative exit data
• Model could improve with hyperparameter tuning
• Health score weights are heuristic-based

NEXT STEPS FOR PRODUCTION
─────────────────────────
• A/B test retention interventions
• Collect exit survey data
• Build longitudinal dataset
• Implement real-time scoring pipeline
""")

---

## What I Learned

This project reinforced several things:

1. **Simple insights often beat complex models.** The correlation analysis and health score are more immediately actionable than the ML model.

2. **Business context matters.** Understanding *why* customers churn is more valuable than just predicting *if* they will.

3. **Segmentation enables targeted action.** You can't treat all customers the same - different groups need different approaches.

4. **Implementation is harder than analysis.** The recommendations are the easy part; execution requires coordination across teams.

5. **Data quality is crucial.** The clean dataset made this analysis smoother than real-world projects often are.

---